In [15]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr


def compute_loss_influence_summary(
    df,
    loss_col="Training_Loss",
    influence_col="Influence",
    label_col="label",
    top_fraction=0.10
):
    """
    Compute the statistical summary requested for the loss–influence analysis.

    Returns:
        dict containing:
        - Spearman(loss, raw influence)
        - Spearman(loss, absolute influence)
        - Proportion of high-loss samples in positive top-10%
        - Proportion of high-loss samples in negative top-10%
        - Proportion of high-loss samples in absolute top-10%
    """

    required = {loss_col, influence_col, label_col}
    missing = required.difference(df.columns)

    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    work = (
        df[[loss_col, influence_col, label_col]]
        .dropna()
        .copy()
    )
    n = len(work)

    if n == 0:
        raise ValueError("No valid samples remain after removing missing values.")

    k = max(1, int(np.ceil(n * top_fraction)))

    # 1. Spearman correlation: loss vs raw signed influence
    rho_raw, p_raw = spearmanr(
        work[loss_col],
        work[influence_col]
    )

    work_neg = work[work[influence_col] < 0].copy()
    
    work_neg["Abs_Influence"] = work_neg[influence_col].abs()
    
    rho_neg_abs, p_neg_abs = spearmanr(
        work_neg[loss_col],
        work_neg["Abs_Influence"]
    )
    
    # -------------------------
    # Positive influence (use raw score)
    # -------------------------
    work_pos = work[work[influence_col] > 0]
    
    rho_pos, p_pos = spearmanr(
        work_pos[loss_col],
        work_pos[influence_col]
    )
        

    work_y0 = work[work[label_col] == 0]

    rho_y0, p_y0 = spearmanr(
        work_y0[loss_col],
        work_y0[influence_col]
    )
    
    # Spearman correlation within label y = 1
    work_y1 = work[work[label_col] == 1]
    
    rho_y1, p_y1 = spearmanr(
        work_y1[loss_col],
        work_y1[influence_col]
    )

    # 2. Spearman correlation: loss vs influence magnitude
    work["Abs_Influence"] = work[influence_col].abs()

    rho_abs, p_abs = spearmanr(
        work[loss_col],
        work["Abs_Influence"]
    )

    # High-loss samples: top 10% by loss
    high_loss_ids = set(
        work.nlargest(k, loss_col).index
    )

    # Top 10% largest positive/raw influence scores
    positive_top_ids = set(
        work.nlargest(k, influence_col).index
    )

    # Top 10% most negative influence scores
    negative_top_ids = set(
        work.nsmallest(k, influence_col).index
    )

    # Top 10% largest influence magnitudes
    absolute_top_ids = set(
        work.nlargest(k, "Abs_Influence").index
    )

    # Denominator is the number of high-loss samples
    high_loss_count = len(high_loss_ids)

    prop_positive = (
        len(high_loss_ids & positive_top_ids)
        / high_loss_count
    )

    prop_negative = (
        len(high_loss_ids & negative_top_ids)
        / high_loss_count
    )

    prop_absolute = (
        len(high_loss_ids & absolute_top_ids)
        / high_loss_count
    )

    return {
        "Spearman(loss, I)": rho_raw,
        "Spearman p-value (raw)": p_raw,
    
        "Spearman(loss, I | y=0)": rho_y0,
        "Spearman p-value (y=0)": p_y0,
    
        "Spearman(loss, I | y=1)": rho_y1,
        "Spearman p-value (y=1)": p_y1,
    
        "Spearman(loss, |I|)": rho_abs,
        "Spearman p-value (abs)": p_abs,

        "Spearman(loss, |I| | I<0)": rho_neg_abs,
        "Spearman p-value (|I| | I<0)": p_neg_abs,
        
        "Spearman(loss, I | I>0)": rho_pos,
        "Spearman p-value (I | I>0)": p_pos,
    
        "High-loss in Pos. Top-10%": prop_positive,
        "High-loss in Neg. Top-10%": prop_negative,
        "High-loss in Abs. Top-10%": prop_absolute,
    
        "N": n,
        "K": k
    }

In [16]:
# Read files
loss_df = pd.read_csv("Train_Loss_Train_Set_1.csv")
if_df   = pd.read_csv("IF_Train_Set_1.csv")
tc_df   = pd.read_csv("TC_Train_Set_1.csv")
labels = pd.read_csv("train_labels.csv")

# Merge
if_data = loss_df.merge(
    if_df[["Train_ID", "Score"]],
    on="Train_ID"
).merge(labels, on="Train_ID", how="left").rename(columns={
    "Loss": "Training_Loss",
    "Score": "Influence"
})

tc_data = loss_df.merge(
    tc_df[["Train_ID", "Score"]],
    on="Train_ID"
).merge(labels, on="Train_ID", how="left").rename(columns={
    "Loss": "Training_Loss",
    "Score": "Influence"
})

In [17]:
# Read files
dia_loss_df = pd.read_csv("Diamonds_Train_Loss_Train_Set_1.csv")
dia_if_df   = pd.read_csv("Diamonds_IF_Train_Set_1.csv")
dia_tc_df   = pd.read_csv("Diamonds_TC_Train_Set_1.csv")
dia_labels = pd.read_csv("train_labels_diamonds.csv")

# Merge
dia_if_data = dia_loss_df.merge(
    dia_if_df[["Train_ID", "Score"]],
    on="Train_ID"
).merge(dia_labels, on="Train_ID", how="left").rename(columns={
    "Loss": "Training_Loss",
    "Score": "Influence"
})

dia_tc_data = dia_loss_df.merge(
    dia_tc_df[["Train_ID", "Score"]],
    on="Train_ID"
).merge(dia_labels, on="Train_ID", how="left").rename(columns={
    "Loss": "Training_Loss",
    "Score": "Influence"
})

In [18]:
datasets = {
    "SynA FOIF": if_data,
    "SynA TC": tc_data,
    "Dia_FOIF":dia_if_data,
    "Dia_TracIn":dia_tc_data,

}

rows = []

for method, df in datasets.items():

    result = compute_loss_influence_summary(df)

    rows.append({
        "Method": method,
        "ρ(loss, I)": result["Spearman(loss, I)"],
        "ρ(loss, I | y=0)": result["Spearman(loss, I | y=0)"],
        "ρ(loss, I | y=1)": result["Spearman(loss, I | y=1)"],
        "ρ(loss, |I|)": result["Spearman(loss, |I|)"],
        "ρ(loss, |I| | I<0)": result["Spearman(loss, |I| | I<0)"],
        "ρ(loss, I | I>0)": result["Spearman(loss, I | I>0)"],
        "High-loss in Pos. Top10%": result["High-loss in Pos. Top-10%"],
        "High-loss in Neg. Top10%": result["High-loss in Neg. Top-10%"],
        "High-loss in Abs. Top10%": result["High-loss in Abs. Top-10%"],
    })

summary = pd.DataFrame(rows)

print(summary)

       Method  ρ(loss, I)  ρ(loss, I | y=0)  ρ(loss, I | y=1)  ρ(loss, |I|)  \
0   SynA FOIF    0.436403          0.462034          0.411593      0.866670   
1     SynA TC   -0.008686          0.700187         -0.691742      0.668394   
2    Dia_FOIF    0.824888          0.758139          0.885361      0.944509   
3  Dia_TracIn    0.111996          0.999006         -0.990948      0.960894   

   ρ(loss, |I| | I<0)  ρ(loss, I | I>0)  High-loss in Pos. Top10%  \
0            0.932740          0.888169                  0.015000   
1            0.691742          0.700187                  0.305000   
2            0.920306          0.970896                  0.474177   
3            0.990950          0.999006                  0.578219   

   High-loss in Neg. Top10%  High-loss in Abs. Top10%  
0                  0.846000                  0.592000  
1                  0.445000                  0.531000  
2                  0.255052                  0.565120  
3                  0.421781       

In [19]:
# check = if_data.copy()

# check["Loss_Decile"] = pd.qcut(
#     check["Training_Loss"],
#     q=10,
#     labels=False,
#     duplicates="drop"
# )

# decile_summary = (
#     check.groupby("Loss_Decile")
#     .apply(
#         lambda x: pd.Series({
#             "Count": len(x),
#             "Mean_Loss": x["Training_Loss"].mean(),
#             "Median_Loss": x["Training_Loss"].median(),
#             "Mean_Influence": x["Influence"].mean(),
#             "Median_Influence": x["Influence"].median(),
#             "Positive_Fraction": (x["Influence"] > 0).mean(),
#             "Spearman_rho": spearmanr(
#                 x["Training_Loss"],
#                 x["Influence"]
#             ).statistic,
#             "Spearman_p": spearmanr(
#                 x["Training_Loss"],
#                 x["Influence"]
#             ).pvalue
#         })
#     )
#     .reset_index()
# )

# print(decile_summary)

In [20]:
# check = dia_if_data.copy()

# check["Loss_Decile"] = pd.qcut(
#     check["Training_Loss"],
#     q=10,
#     labels=False,
#     duplicates="drop"
# )

# decile_summary = (
#     check.groupby("Loss_Decile")
#     .apply(
#         lambda x: pd.Series({
#             "Count": len(x),
#             "Mean_Loss": x["Training_Loss"].mean(),
#             "Median_Loss": x["Training_Loss"].median(),
#             "Mean_Influence": x["Influence"].mean(),
#             "Median_Influence": x["Influence"].median(),
#             "Positive_Fraction": (x["Influence"] > 0).mean(),
#             "Spearman_rho": spearmanr(
#                 x["Training_Loss"],
#                 x["Influence"]
#             ).statistic,
#             "Spearman_p": spearmanr(
#                 x["Training_Loss"],
#                 x["Influence"]
#             ).pvalue
#         })
#     )
#     .reset_index()
# )

# print(decile_summary)

In [21]:
# check = tc_data.copy()

# check["Loss_Decile"] = pd.qcut(
#     check["Training_Loss"],
#     q=10,
#     labels=False,
#     duplicates="drop"
# )

# decile_summary = (
#     check.groupby("Loss_Decile")
#     .apply(
#         lambda x: pd.Series({
#             "Count": len(x),
#             "Mean_Loss": x["Training_Loss"].mean(),
#             "Median_Loss": x["Training_Loss"].median(),
#             "Mean_Influence": x["Influence"].mean(),
#             "Median_Influence": x["Influence"].median(),
#             "Positive_Fraction": (x["Influence"] > 0).mean(),
#             "Spearman_rho": spearmanr(
#                 x["Training_Loss"],
#                 x["Influence"]
#             ).statistic,
#             "Spearman_p": spearmanr(
#                 x["Training_Loss"],
#                 x["Influence"]
#             ).pvalue
#         })
#     )
#     .reset_index()
# )

# print(decile_summary)